In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ==========================================
# 1. Load Dataset
# ==========================================

df = pd.read_csv(
    "data/market_data_final_features.csv",
    parse_dates=["Date"],
    index_col="Date"
)

df = df.dropna(
    subset=["HMM_Regime_Risk"]
).copy()

TARGET = "Crash_Risk"


# ==========================================
# 2. Market + TDA Features ONLY
# ==========================================

exclude = [
    TARGET,
    "Future_Min_SP500",
    "Future_Drawdown_10D",
    "Market_Regime",
    "HMM_State",
    "HMM_Regime_Risk",

    # Remove PELT
    "PELT_Recent_Change",
    "PELT_Days_Since_Change",
    "PELT_Change_Strength",
    "PELT_Change_30D",
    "PELT_Change_60D",
    "PELT_Recency_Score",
    "PELT_Structural_Stress",

    # Raw price levels
    "SP500",
    "NASDAQ",
    "GOLD",
    "OIL",
    "SP500_MA20",
    "SP500_MA50"
]

features = [
    col for col in df.columns
    if col not in exclude
]

print("Feature Count:", len(features))

print("\nFeatures:")
for col in features:
    print(col)


# ==========================================
# 3. Chronological Split
# ==========================================

split = int(len(df) * 0.80)

X = df[features]
y = df[TARGET]

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]


print(
    "\nTrain Period:",
    X_train.index.min(),
    "to",
    X_train.index.max()
)

print(
    "Test Period:",
    X_test.index.min(),
    "to",
    X_test.index.max()
)


# ==========================================
# 4. Class Weight
# ==========================================

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

weight = negative / positive

print(
    "\nScale Pos Weight:",
    round(weight, 2)
)


# ==========================================
# 5. Train Final Market + TDA Model
# ==========================================

model = CatBoostClassifier(

    iterations=700,
    depth=6,
    learning_rate=0.03,

    loss_function="Logloss",
    eval_metric="AUC",

    scale_pos_weight=weight,

    random_seed=42,

    verbose=False,

    allow_writing_files=False
)

model.fit(

    X_train,
    y_train,

    eval_set=(
        X_test,
        y_test
    ),

    early_stopping_rounds=100,

    verbose=False
)


# ==========================================
# 6. Prediction Probabilities
# ==========================================

prob = model.predict_proba(
    X_test
)[:, 1]


print(
    "\nROC-AUC:",
    round(
        roc_auc_score(
            y_test,
            prob
        ),
        4
    )
)

print(
    "PR-AUC:",
    round(
        average_precision_score(
            y_test,
            prob
        ),
        4
    )
)


# ==========================================
# 7. Threshold Search
# ==========================================

results = []

for threshold in np.arange(
    0.05,
    0.951,
    0.01
):

    pred = (
        prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred,
        zero_division=0
    )

    tn, fp, fn, tp = (
        confusion_matrix(
            y_test,
            pred,
            labels=[0, 1]
        ).ravel()
    )

    results.append({

        "Threshold":
            round(threshold, 2),

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,

        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


results_df = pd.DataFrame(
    results
)


# ==========================================
# 8. Best F1 Threshold
# ==========================================

best_f1 = results_df.loc[
    results_df["F1"].idxmax()
]

print(
    "\n========== BEST F1 THRESHOLD =========="
)

print(
    best_f1.to_string()
)


# ==========================================
# 9. High Recall Threshold
# ==========================================

# Find thresholds achieving >= 80% recall

high_recall = results_df[
    results_df["Recall"] >= 0.80
]

if not high_recall.empty:

    best_high_recall = (
        high_recall
        .sort_values(
            "Precision",
            ascending=False
        )
        .iloc[0]
    )

    print(
        "\n========== >=80% RECALL OPTION =========="
    )

    print(
        best_high_recall.to_string()
    )

else:

    print(
        "\nNo threshold achieved 80% recall."
    )


# ==========================================
# 10. Top Thresholds
# ==========================================

print(
    "\n========== TOP 10 BY F1 =========="
)

print(

    results_df

    .sort_values(
        "F1",
        ascending=False
    )

    .head(10)

    .to_string(
        index=False
    )

)


# ==========================================
# 11. Save
# ==========================================

results_df.to_csv(
    "data/final_threshold_analysis.csv",
    index=False
)

model.save_model(
    "market_tda_final_model.cbm"
)


# Save feature list

pd.Series(
    features,
    name="Feature"
).to_csv(
    "data/final_model_features.csv",
    index=False
)


# Save test predictions

prediction_df = pd.DataFrame({

    "Actual":
        y_test,

    "Crash_Probability":
        prob

})

prediction_df.to_csv(
    "data/final_test_predictions.csv"
)


print(
    "\nSaved final model + threshold results."
)

Feature Count: 29

Features:
VIX
SP500_Return
NASDAQ_Return
GOLD_Return
OIL_Return
SP500_Log_Return
NASDAQ_Log_Return
GOLD_Log_Return
OIL_Log_Return
SP500_Volatility_10
SP500_Volatility_20
NASDAQ_Volatility_20
SP500_Momentum_5
SP500_Momentum_10
SP500_Momentum_20
SP500_MA20_Distance
SP500_MA50_Distance
VIX_Change
VIX_MA10
VIX_Ratio
TDA_H0_Count
TDA_H0_TotalPersistence
TDA_H0_MaxPersistence
TDA_H0_MeanPersistence
TDA_H1_Count
TDA_H1_TotalPersistence
TDA_H1_MaxPersistence
TDA_H1_MeanPersistence
TDA_H1_Entropy

Train Period: 2002-12-19 00:00:00 to 2021-10-14 00:00:00
Test Period: 2021-10-15 00:00:00 to 2026-07-02 00:00:00

Scale Pos Weight: 26.55

ROC-AUC: 0.8365
PR-AUC: 0.1547

========== BEST F1 THRESHOLD ==========
Threshold       0.620000
Precision       0.206522
Recall          0.463415
F1              0.285714
TP             19.000000
FP             73.000000
FN             22.000000
TN           1071.000000

========== >=80% RECALL OPTION ==========
Threshold      0.370000
Precision

In [2]:
import pandas as pd

df = pd.read_csv(
    "data/final_threshold_analysis.csv"
)

best = df.loc[df["F1"].idxmax()]

print("\nBEST F1 THRESHOLD")
print(best)

high_recall = df[df["Recall"] >= 0.80]

if not high_recall.empty:
    best_recall = high_recall.sort_values(
        "Precision",
        ascending=False
    ).iloc[0]

    print("\nBEST >=80% RECALL")
    print(best_recall)


BEST F1 THRESHOLD
Threshold       0.620000
Precision       0.206522
Recall          0.463415
F1              0.285714
TP             19.000000
FP             73.000000
FN             22.000000
TN           1071.000000
Name: 57, dtype: float64

BEST >=80% RECALL
Threshold      0.370000
Precision      0.141667
Recall         0.829268
F1             0.241993
TP            34.000000
FP           206.000000
FN             7.000000
TN           938.000000
Name: 32, dtype: float64
